In [135]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('Dataset'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Dataset\gender_submission.csv
Dataset\test.csv
Dataset\train.csv


In [136]:
train = pd.read_csv(r'Dataset\train.csv')
test = pd.read_csv(r'Dataset\test.csv')
train.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


## Clean the Data. 

- see how many null values does the data have: <br>
```

print('Missing values Percentage: \n\n', round(df.isnull().sum().sort_values(ascending=False)/len(df)*100,1))

In [137]:
print('Missing values Percentage: \n\n', round (train.isnull().sum().sort_values(ascending=False)/len(train)*100,1))
print('Missing values Percentage: \n\n', round (test.isnull().sum().sort_values(ascending=False)/len(test)*100,1))


Missing values Percentage: 

 Cabin          77.1
Age            19.9
Embarked        0.2
PassengerId     0.0
Survived        0.0
Pclass          0.0
Name            0.0
Sex             0.0
SibSp           0.0
Parch           0.0
Ticket          0.0
Fare            0.0
dtype: float64
Missing values Percentage: 

 Cabin          78.2
Age            20.6
Fare            0.2
PassengerId     0.0
Pclass          0.0
Name            0.0
Sex             0.0
SibSp           0.0
Parch           0.0
Ticket          0.0
Embarked        0.0
dtype: float64


we see that the titles can be category of guesing the age of a person. like the title master. which can only be used for boys. and now we are going to extract the title and use it as one of hte methods filling up the age with mean values base on class, sex, parent (if they have or none) and title 

In [138]:
from sklearn.model_selection import train_test_split


y = train.Survived
train.drop(['Survived'], axis=1, inplace=True)


X_train, X_valid, y_train, y_valid = train_test_split(train, y, train_size=0.8, test_size=0.2, random_state= 0)


X_train['Titles'] = X_train['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
X_valid['Titles'] = X_valid['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test['Titles'] = test['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)


In [139]:
X_train.shape

(712, 12)

In [140]:
display(X_train.groupby(['Sex', 'Pclass', 'Parch', 'Titles'])['Age'].mean())

Sex     Pclass  Parch  Titles  
female  1       0      Countess    33.000000
                       Lady        48.000000
                       Miss        36.400000
                       Mlle        24.000000
                       Mme         24.000000
                       Mrs         39.470588
                1      Miss        21.500000
                       Mrs         47.375000
                2      Miss        19.888889
                       Mrs         30.500000
        2       0      Miss        29.633333
                       Mrs         34.055556
                       Ms          28.000000
                1      Miss        13.200000
                       Mrs         34.500000
                2      Miss        11.600000
                       Mrs         34.000000
                3      Mrs         39.000000
        3       0      Miss        21.575758
                       Mrs         29.142857
                1      Miss         5.159091
                       

we need to group titles like captain, col, major as one in order to remove sparsity. where a model learns less to feature that appears rarely. 

In [141]:
TitleDict = {"Capt": "Officer","Col": "Officer","Major": "Officer","Jonkheer": "Royalty", \
             "Don": "Royalty", "Sir" : "Royalty","Dr": "Royalty","Rev": "Royalty", \
             "Countess":"Royalty", "Mme": "Mrs", "Mlle": "Miss", "Ms": "Mrs","Mr" : "Mr", \
             "Mrs" : "Mrs","Miss" : "Miss","Master" : "Master","Lady" : "Royalty"}

X_train['Titles'] = X_train.Titles.map(TitleDict)
X_valid['Titles'] = X_valid.Titles.map(TitleDict)
test['Titles'] = test.Titles.map(TitleDict)

In [142]:
def columnCategory(dataset, categoryTotal=20):
    """
    Displays an orderly breakdown of every column. 
    Lists specific values for low cardinality columns, and flags 
    high cardinality columns without flooding the screen.
    """
    print(f"=== DATAFRAME CARDINALITY & CATEGORY ANALYSIS ===")
    print(f" Threshold for Low Cardinality: <= {categoryTotal} unique values\n")
    print("=" * 60)
    
    for column in dataset.columns:
        # Get unique values and ignore null values for an accurate count
        unique_vals = dataset[column].dropna().unique()
        unique_count = len(unique_vals)
        
        print(f"📌 Column: {column}")
        print(f"   • Total Unique Values: {unique_count}")
        
        # Condition A: Low Cardinality (Categorical)
        if unique_count <= categoryTotal:
            print(f"   • Status: [ LOW CARDINALITY / CATEGORY ]")
            # Sort the unique values for an orderly display
            try:
                sorted_vals = sorted(unique_vals)
            except TypeError:
                sorted_vals = sorted(unique_vals, key=str)
                
            print(f"   • Categories: {sorted_vals}")
            
        # Condition B: High Cardinality
        else:
            print(f"   • Status: [ HIGH CARDINALITY ]")
            print(f"   • Note: Exceeds threshold. Likely an ID, Name, or continuous numeric column.")
            
        print("-" * 60)

columnCategory(X_train)

=== DATAFRAME CARDINALITY & CATEGORY ANALYSIS ===
 Threshold for Low Cardinality: <= 20 unique values

📌 Column: PassengerId
   • Total Unique Values: 712
   • Status: [ HIGH CARDINALITY ]
   • Note: Exceeds threshold. Likely an ID, Name, or continuous numeric column.
------------------------------------------------------------
📌 Column: Pclass
   • Total Unique Values: 3
   • Status: [ LOW CARDINALITY / CATEGORY ]
   • Categories: [np.int64(1), np.int64(2), np.int64(3)]
------------------------------------------------------------
📌 Column: Name
   • Total Unique Values: 712
   • Status: [ HIGH CARDINALITY ]
   • Note: Exceeds threshold. Likely an ID, Name, or continuous numeric column.
------------------------------------------------------------
📌 Column: Sex
   • Total Unique Values: 2
   • Status: [ LOW CARDINALITY / CATEGORY ]
   • Categories: ['female', 'male']
------------------------------------------------------------
📌 Column: Age
   • Total Unique Values: 86
   • Status: [ HI

In [143]:
display(X_valid.groupby(['Sex','Titles', 'Pclass', 'Parch'])['Age'].mean())

Sex     Titles   Pclass  Parch
female  Miss     1       0        31.777778
                         1        22.000000
                         2        24.500000
                 2       0        33.750000
                         1         3.000000
                         2         7.000000
                 3       0        22.100000
                         1         0.750000
                         2         7.333333
        Mrs      1       0        36.250000
                         1        44.666667
                 2       0        31.600000
                         1        32.000000
                         2        24.000000
                 3       0        40.000000
                         1        36.333333
                         2        28.000000
                         4        29.000000
                         5        41.000000
        Royalty  1       0        49.000000
male    Master   2       1         3.000000
                 3       1         5.473333
 

In [144]:
# GROUP AND FIND THE AVERAGE 

master_means = X_train.groupby(['Sex','Titles', 'Pclass', 'Parch'])['Age'].mean().round(2)
general_means = X_train.groupby(['Sex', 'Pclass', 'Parch'])['Age'].mean().round(2)
global_means = X_train['Age'].mean().round(2)

def FindingAverage(dataset, master, general, global_means):
    # Match the current dataset's rows to the master lookup map indices
    master_means = dataset.set_index(['Sex', 'Titles', 'Pclass', 'Parch']).index.map(master).to_series()
    master_means.index = dataset.index # Keep indices perfectly aligned with the current dataset
    
    # Match the current dataset's rows to the general lookup map indices
    general_means = dataset.set_index(['Sex', 'Pclass', 'Parch']).index.map(general).to_series()
    general_means.index = dataset.index

    dataset['Age'] = dataset['Age'].fillna(master_means)
    dataset['Age'] = dataset['Age'].fillna(general_means)
    dataset['Age'] = dataset['Age'].fillna(global_means)

    return dataset


X_train = FindingAverage(X_train, master_means,general_means, global_means)
X_valid = FindingAverage(X_valid, master_means,general_means, global_means)
test = FindingAverage(test, master_means,general_means, global_means)

In [145]:
train.sample(4)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
157,158,3,"Corn, Mr. Harry",male,30.0,0,0,SOTON/OQ 392090,8.05,NaN,S
520,521,1,"Perreault, Miss. Anne",female,30.0,0,0,12749,93.50,B73,S
633,634,1,"Parr, Mr. William Henry Marsh",male,NaN,0,0,112052,0.00,NaN,S
708,709,1,"Cleaver, Miss. Alice",female,22.0,0,0,113781,151.55,NaN,S


In [146]:
print('Missing values Percentage: \n\n', round (X_train.isnull().sum().sort_values(ascending=False)/len(X_train)*100,1))
print('Missing values Percentage: \n\n', round (X_valid.isnull().sum().sort_values(ascending=False)/len(X_valid)*100,1))
print('Missing values Percentage: \n\n', round (test.isnull().sum().sort_values(ascending=False)/len(test)*100,1))

Missing values Percentage: 

 Cabin          77.1
Embarked        0.3
PassengerId     0.0
Pclass          0.0
Name            0.0
Sex             0.0
Age             0.0
SibSp           0.0
Parch           0.0
Ticket          0.0
Fare            0.0
Titles          0.0
dtype: float64
Missing values Percentage: 

 Cabin          77.1
PassengerId     0.0
Pclass          0.0
Name            0.0
Sex             0.0
Age             0.0
SibSp           0.0
Parch           0.0
Ticket          0.0
Fare            0.0
Embarked        0.0
Titles          0.0
dtype: float64
Missing values Percentage: 

 Cabin          78.2
Fare            0.2
Titles          0.2
PassengerId     0.0
Pclass          0.0
Name            0.0
Sex             0.0
Age             0.0
SibSp           0.0
Parch           0.0
Ticket          0.0
Embarked        0.0
dtype: float64


lets fill up the remaining missing parts

In [147]:
X_train['Embarked'] = X_train['Embarked'].fillna('U')
X_valid['Embarked'] = X_valid['Embarked'].fillna('U')

fillFare = test['Fare'].mean().round(2)
test['Fare'] = test['Fare'].fillna(fillFare)
X_train = X_train.drop(['Ticket', 'Cabin', 'PassengerId', 'Name'], axis=1)
X_valid = X_valid.drop(['Ticket', 'Cabin', 'PassengerId', 'Name'], axis=1)
test = test.drop(['Ticket', 'Cabin', 'PassengerId', 'Name'], axis=1)
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Titles
140,3,female,32.5,0,2,15.2458,C,Mrs
439,2,male,31.0,0,0,10.5000,S,Mr
817,2,male,31.0,1,1,37.0042,C,Mr
378,3,male,20.0,0,0,4.0125,C,Mr
491,3,male,21.0,0,0,7.2500,S,Mr


In [148]:
print('Missing values Percentage: \n\n', round (X_train.isnull().sum().sort_values(ascending=False)/len(X_train)*100,1))
print('Missing values Percentage: \n\n', round (X_valid.isnull().sum().sort_values(ascending=False)/len(X_valid)*100,1))
print('Missing values Percentage: \n\n', round (test.isnull().sum().sort_values(ascending=False)/len(test)*100,1))


Missing values Percentage: 

 Pclass      0.0
Sex         0.0
Age         0.0
SibSp       0.0
Parch       0.0
Fare        0.0
Embarked    0.0
Titles      0.0
dtype: float64
Missing values Percentage: 

 Pclass      0.0
Sex         0.0
Age         0.0
SibSp       0.0
Parch       0.0
Fare        0.0
Embarked    0.0
Titles      0.0
dtype: float64
Missing values Percentage: 

 Titles      0.2
Pclass      0.0
Sex         0.0
Age         0.0
SibSp       0.0
Parch       0.0
Fare        0.0
Embarked    0.0
dtype: float64


In [149]:
X_train.shape, X_valid.shape, test.shape

((712, 8), (179, 8), (418, 8))

In [150]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Titles
140,3,female,32.5,0,2,15.2458,C,Mrs
439,2,male,31.0,0,0,10.5000,S,Mr
817,2,male,31.0,1,1,37.0042,C,Mr
378,3,male,20.0,0,0,4.0125,C,Mr
491,3,male,21.0,0,0,7.2500,S,Mr


# Look into the cardinality. 

for columns that are cateogorical (less than 10 categorical. a low cardinality) lets use one hot encoded for them 

In [151]:
# "Cardinality" means the number of unique values in a column
# Select categorical columns with relatively low cardinality (convenient but arbitrary)
low_cardinality_cols = [cname for cname in X_train.columns if X_train[cname].nunique() < 10 and 
                        X_train[cname].dtype == "object"]

# Select numeric columns
numeric_cols = [cname for cname in X_train.columns if X_train[cname].dtype in ['int64', 'float64']]


# Keep selected columns only
my_cols = low_cardinality_cols + numeric_cols
X_train = X_train[my_cols].copy()
X_valid = X_valid[my_cols].copy()
test = test[my_cols].copy()

# One-hot encode the data (to shorten the code, we use pandas)
X_train = pd.get_dummies(X_train)
X_valid = pd.get_dummies(X_valid)
test = pd.get_dummies(test)
X_train, X_valid = X_train.align(X_valid, join='left', axis=1)
X_valid = X_valid.fillna(0)  # Fill any newly added category columns with 0
X_train, test = X_train.align(test, join='left', axis=1)
test = test.fillna(0)  # Fill any newly added category columns with 0

In [152]:
print('Missing values Percentage: \n\n', round (test.isnull().sum().sort_values(ascending=False)/len(test)*100,1))

Missing values Percentage: 

 Pclass            0.0
Embarked_S        0.0
Titles_Officer    0.0
Titles_Mrs        0.0
Titles_Mr         0.0
Titles_Miss       0.0
Titles_Master     0.0
Embarked_U        0.0
Embarked_Q        0.0
Age               0.0
Embarked_C        0.0
Sex_male          0.0
Sex_female        0.0
Fare              0.0
Parch             0.0
SibSp             0.0
Titles_Royalty    0.0
dtype: float64


In [153]:
X_train.shape

(712, 17)

In [154]:
X_train.head()

,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_U,Titles_Master,Titles_Miss,Titles_Mr,Titles_Mrs,Titles_Officer,Titles_Royalty
140,3,32.5,0,2,15.2458,True,False,True,False,False,False,False,False,False,True,False,False
439,2,31.0,0,0,10.5000,False,True,False,False,True,False,False,False,True,False,False,False
817,2,31.0,1,1,37.0042,False,True,True,False,False,False,False,False,True,False,False,False
378,3,20.0,0,0,4.0125,False,True,True,False,False,False,False,False,True,False,False,False
491,3,21.0,0,0,7.2500,False,True,False,False,True,False,False,False,True,False,False,False


In [155]:
print(X_train['Sex_female'].dtype)

bool


# XGBoost

for this model we are going to use an XGClassifier model. XGboost training compared to a random forest training Sequentially. It builds one shallow tree, calculates its errors, and builds the next tree specifically to correct those mistakes.

In [ ]:
from xgboost import XGBClassifier()

# Define the model
my_model_1 = XGBRegressor(random_state=0) # Your code here

# Fit the model
my_model_1.fit(X_train, y_train) # Your code here

# Check your answer